# 🛡️ VisionGuard: Reliable Colab EfficientAD Trainer (Drive Auto-Sync)

### Fixes Applied:
1. **`train_batch_size = 1`**: EfficientAD requires batch size 1 (by architecture design).
2. **Reliable Cloud Datasets**:
   - **VisA**: Automatically downloaded via official Amazon S3 repository.
   - **MVTec AD**: Automatically downloaded via official MVTec high-speed tarball with `wget` and extracted on-demand.
3. **Automatic Google Drive Sync**: Every `.pt` checkpoint (~15 MB) is saved to `MyDrive/VisionGuard_Checkpoints/` immediately after training.

## Step 1: Verify T4 GPU

In [ ]:
!nvidia-smi
import torch
if not torch.cuda.is_available():
    raise SystemError("Please set Runtime -> Change runtime type -> T4 GPU")
print("✅ Connected to:", torch.cuda.get_device_name(0))

## Step 2: Mount Google Drive

In [ ]:
from google.colab import drive
import os
from pathlib import Path

drive.mount('/content/drive')
DRIVE_DIR = Path("/content/drive/MyDrive/VisionGuard_Checkpoints")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print("✅ Checkpoints will save to:", DRIVE_DIR)

## Step 3: Install Anomalib 2.7.0 & Dependencies

In [ ]:
!pip install -q --upgrade pip
!pip install -q "anomalib==2.7.0" "torch>=2.0.0" "torchvision>=0.15.0" opencv-python scikit-learn pandas gdown

## Step 4: Download MVTec AD Dataset Directly (Bypasses Broken CDN)

In [ ]:
import os
from pathlib import Path

MVTEC_DIR = Path("./datasets/mvtec_ad")
MVTEC_DIR.mkdir(parents=True, exist_ok=True)

# Check if already downloaded
if not (MVTEC_DIR / "bottle").exists():
    print("Downloading official MVTec AD dataset archive (~4.9 GB) directly...")
    !wget -q --show-progress -c https://www.mydrive.ch/shares/38536/38307f5f7f1e16427b1ec770e0ea11f8/download/420938113-1629952094/mvtec_anomaly_detection.tar.xz
    print("Extracting MVTec AD archive...")
    !tar -xf mvtec_anomaly_detection.tar.xz -C ./datasets/mvtec_ad
    !rm mvtec_anomaly_detection.tar.xz
    print("✅ MVTec AD extracted successfully!")
else:
    print("✅ MVTec AD already present on disk.")

## Step 5: Automated Batch Training Loop (EfficientAD -> Google Drive)
- Uses **`train_batch_size = 1`** as required by EfficientAD architecture.
- Automatically iterates over all categories across both datasets.
- Auto-skips any category already saved in your Google Drive.

In [ ]:
import gc
import shutil
import json
import torch
from anomalib.models import Efficientad
from anomalib.engine import Engine
from anomalib.data import MVTecAD, Visa

ALL_JOBS = [
    # --- 1. Amazon VisA (12 Categories) ---
    {"dataset": "visa", "category": "pcb1"},
    {"dataset": "visa", "category": "pcb2"},
    {"dataset": "visa", "category": "pcb3"},
    {"dataset": "visa", "category": "pcb4"},
    {"dataset": "visa", "category": "candle"},
    {"dataset": "visa", "category": "capsules"},
    {"dataset": "visa", "category": "cashew"},
    {"dataset": "visa", "category": "chewinggum"},
    {"dataset": "visa", "category": "fryum"},
    {"dataset": "visa", "category": "macaroni1"},
    {"dataset": "visa", "category": "macaroni2"},
    {"dataset": "visa", "category": "pipe_fryum"},

    # --- 2. MVTec AD (15 Categories) ---
    {"dataset": "mvtec", "category": "bottle"},
    {"dataset": "mvtec", "category": "cable"},
    {"dataset": "mvtec", "category": "capsule"},
    {"dataset": "mvtec", "category": "carpet"},
    {"dataset": "mvtec", "category": "grid"},
    {"dataset": "mvtec", "category": "hazelnut"},
    {"dataset": "mvtec", "category": "leather"},
    {"dataset": "mvtec", "category": "metal_nut"},
    {"dataset": "mvtec", "category": "pill"},
    {"dataset": "mvtec", "category": "screw"},
    {"dataset": "mvtec", "category": "tile"},
    {"dataset": "mvtec", "category": "toothbrush"},
    {"dataset": "mvtec", "category": "transistor"},
    {"dataset": "mvtec", "category": "wood"},
    {"dataset": "mvtec", "category": "zipper"}
]

total = len(ALL_JOBS)

for idx, job in enumerate(ALL_JOBS, start=1):
    dataset_type = job["dataset"]
    category = job["category"]
    
    dest_ckpt = DRIVE_DIR / f"efficientad_{category}.pt"
    dest_report = DRIVE_DIR / f"eval_efficientad_{category}.json"
    
    if dest_ckpt.exists():
        print(f"\n[{idx}/{total}] SKIP: '{category}' already in Google Drive.")
        continue
        
    print(f"\n=======================================================")
    print(f"[{idx}/{total}] TRAINING: {dataset_type.upper()} - {category.upper()}")
    print(f"=======================================================")
    
    temp_run_dir = Path(f"./runs/{category}")
    
    try:
        # NOTE: train_batch_size MUST be 1 for EfficientAD
        if dataset_type == "visa":
            datamodule = Visa(
                root="./datasets/visa",
                category=category,
                train_batch_size=1,
                eval_batch_size=1,
                num_workers=2
            )
        else:
            datamodule = MVTecAD(
                root="./datasets/mvtec_ad",
                category=category,
                train_batch_size=1,
                eval_batch_size=1,
                num_workers=2
            )
        
        model = Efficientad()
        engine = Engine(
            default_root_dir=str(temp_run_dir),
            accelerator="gpu",
            devices=1,
            max_epochs=70
        )
        
        # Train 70 epochs on T4 GPU (~1.5 to 2 mins)
        engine.fit(datamodule=datamodule, model=model)
        
        # Evaluate
        test_results = engine.test(datamodule=datamodule, model=model)
        print(f"Results for {category}:", test_results)
        
        # Save directly to Google Drive
        torch.save(model.state_dict(), dest_ckpt)
        file_size_mb = os.path.getsize(dest_ckpt) / (1024 * 1024)
        print(f"✅ Successfully saved {dest_ckpt.name} to Google Drive ({file_size_mb:.1f} MB)")
        
        with open(dest_report, "w", encoding="utf-8") as f:
            json.dump({"dataset": dataset_type, "category": category, "model": "efficientad", "metrics": str(test_results)}, f, indent=2)
            
    except Exception as e:
        print(f"❌ ERROR on {category}: {e}")
        
    finally:
        shutil.rmtree(temp_run_dir, ignore_errors=True)
        gc.collect()
        torch.cuda.empty_cache()

print("\n=======================================================")
print("ALL EFFICIENTAD MODELS COMPLETED!")
print(f"Saved in Google Drive: MyDrive/VisionGuard_Checkpoints/")
print("=======================================================")

## Step 6: Verify Saved Checkpoints in Google Drive

In [ ]:
print("Your trained EfficientAD checkpoints:")
for f in sorted(DRIVE_DIR.glob("*.pt")):
    print(f" - {f.name} ({f.stat().st_size / (1024*1024):.1f} MB)")